In [1]:
!pip install -q ultralytics tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.5 MB/s eta 0:00:00


In [2]:
import os
import cv2
import pickle
import numpy as np
from ultralytics import YOLO
from pathlib import Path

# ---- CONFIG ----
MULTI_MODEL_PATH = "/kaggle/input/datasets/zeinsaad/ball-detector/multi_model_finetuned_lastversion.pt"   # goalkeeper, player, referee (ball ignored — multi model does not detect ball reliably)
BALL_MODEL_PATH  = "/kaggle/input/datasets/zeinsaad/ball-detector/best_afterfinish.pt"          # dedicated ball detector — sole source of ball detections

VIDEO_PATH = "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/barca_atletico_46m53s.mkv"
OUTPUT_CACHE_PATH = "barca_atletico_firsthalf_detection_cache_final.pkl"

# class name mapping for the multiclass model — set to match YOUR model's id order
MULTI_CLASS_NAMES = {0: "ball", 1: "goalkeeper", 2: "player", 3: "referee"}

CONF_THRESH_MULTI = 0.25
CONF_THRESH_BALL  = 0.15   # ball model usually needs a lower floor
BALL_LOW_CONF_FLAG = 0.25  # below this, mark low_confidence=True (don't drop)

CROSS_CLASS_IOU_THRESH = 0.5   # gk/referee suppresses overlapping player box
SAME_CLASS_NMS_IOU = 0.5

DEVICE = 0   # set "cuda" / 0 on Kaggle

IMG_SIZE = 1280

CHECKPOINT_EVERY = 2000
CHECKPOINT_PATH = "/kaggle/working/detection_cache_ckpt.pkl"

print("Config set. Checking paths...")
for p in [MULTI_MODEL_PATH, BALL_MODEL_PATH, VIDEO_PATH]:
    print(p, "->", "OK" if os.path.exists(p) else "MISSING")


Config set. Checking paths...
/kaggle/input/datasets/zeinsaad/ball-detector/multi_model_finetuned_lastversion.pt -> OK
/kaggle/input/datasets/zeinsaad/ball-detector/best_afterfinish.pt -> OK
/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/barca_atletico_46m53s.mkv -> OK


In [3]:
multi_model = YOLO(MULTI_MODEL_PATH)
ball_model  = YOLO(BALL_MODEL_PATH)

print("Multiclass model classes:", multi_model.names)
print("Ball model classes:", ball_model.names)

# sanity check against your MULTI_CLASS_NAMES mapping above
if multi_model.names != MULTI_CLASS_NAMES:
    print("\n⚠️ WARNING: MULTI_CLASS_NAMES in the config cell does not match model.names.")
    print("Update MULTI_CLASS_NAMES to match exactly before continuing.")
else:
    print("\n✅ Class mapping matches.")


Multiclass model classes: {0: 'ball', 1: 'goalkeeper', 2: 'player', 3: 'referee'}
Ball model classes: {0: 'ball'}

✅ Class mapping matches.


In [4]:
def iou(box1, box2):
    """box format: [x1, y1, x2, y2]"""
    xa1, ya1, xa2, ya2 = box1
    xb1, yb1, xb2, yb2 = box2

    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)
    union = area_a + area_b - inter_area

    return inter_area / union if union > 0 else 0.0


def same_class_nms(dets, iou_thresh):
    """dets: list of dicts with 'bbox','conf','class'. Standard per-class NMS."""
    by_class = {}
    for d in dets:
        by_class.setdefault(d["class"], []).append(d)

    kept = []
    for cls, cls_dets in by_class.items():
        cls_dets = sorted(cls_dets, key=lambda d: d["conf"], reverse=True)
        used = [False] * len(cls_dets)
        for i in range(len(cls_dets)):
            if used[i]:
                continue
            kept.append(cls_dets[i])
            for j in range(i + 1, len(cls_dets)):
                if used[j]:
                    continue
                if iou(cls_dets[i]["bbox"], cls_dets[j]["bbox"]) >= iou_thresh:
                    used[j] = True
    return kept


def cross_class_suppress_players(dets, iou_thresh):
    """
    If a 'player' box overlaps a 'goalkeeper' or 'referee' box above iou_thresh,
    drop the player box (gk/referee wins). No color/classifier logic — pure
    detection-level dedup for the case where the model emits two competing
    boxes for the same person.
    """
    priority_boxes = [d for d in dets if d["class"] in ("goalkeeper", "referee")]
    player_boxes = [d for d in dets if d["class"] == "player"]
    other_boxes = [d for d in dets if d["class"] not in ("goalkeeper", "referee", "player")]

    kept_players = []
    for p in player_boxes:
        suppressed = False
        for pb in priority_boxes:
            if iou(p["bbox"], pb["bbox"]) >= iou_thresh:
                suppressed = True
                break
        if not suppressed:
            kept_players.append(p)

    return priority_boxes + kept_players + other_boxes


def process_ball_detections(ball_model_dets, low_conf_flag):
    """
    Ball candidates come ONLY from the dedicated ball model (multi model is not
    used for ball — it does not detect it reliably). Keep ONLY the single
    highest-confidence candidate if multiple boxes are returned; downstream
    tracking expects at most one ball per frame.
    """
    if not ball_model_dets:
        return []

    best = max(ball_model_dets, key=lambda d: d["conf"])
    best["low_confidence"] = best["conf"] < low_conf_flag
    best["class"] = "ball"

    return [best]

print("Helper functions loaded.")


Helper functions loaded.


In [5]:
import cv2
import pickle
import os
from tqdm.auto import tqdm

# --- resume from checkpoint if one exists ---
start_frame = 0
detection_cache = {}
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "rb") as f:
        ckpt = pickle.load(f)
    detection_cache = ckpt["detection_cache"]
    start_frame = ckpt["frame_idx"] + 1
    print(f"Resuming from frame {start_frame} ({len(detection_cache)} frames already cached)")

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Video: {total_frames} frames @ {fps:.2f} fps")

if start_frame > 0:
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

frame_idx = start_frame
pbar = tqdm(total=total_frames, initial=start_frame, desc="Detecting", unit="frame")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # --- multiclass model inference (goalkeeper / player / referee only — ball skipped) ---
    multi_res = multi_model.predict(frame, conf=CONF_THRESH_MULTI, imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    multi_dets = []
    for box in multi_res.boxes:
        cls_id = int(box.cls[0])
        cls_name = MULTI_CLASS_NAMES[cls_id]
        if cls_name == "ball":
            continue  # ball is sourced only from the dedicated ball model
        conf = float(box.conf[0])
        bbox = box.xyxy[0].tolist()
        multi_dets.append({"bbox": bbox, "conf": conf, "class": cls_name, "source": "multi"})

    # --- ball-only model inference (sole source of ball detections) ---
    ball_res = ball_model.predict(frame, conf=CONF_THRESH_BALL, imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    ball_model_dets = []
    for box in ball_res.boxes:
        conf = float(box.conf[0])
        bbox = box.xyxy[0].tolist()
        ball_model_dets.append({"bbox": bbox, "conf": conf, "class": "ball", "source": "ball_model"})

    fused_ball = process_ball_detections(ball_model_dets, low_conf_flag=BALL_LOW_CONF_FLAG)

    # --- dedup non-ball classes ---
    non_ball_dets = same_class_nms(multi_dets, iou_thresh=SAME_CLASS_NMS_IOU)
    non_ball_dets = cross_class_suppress_players(non_ball_dets, iou_thresh=CROSS_CLASS_IOU_THRESH)

    detection_cache[frame_idx] = fused_ball + non_ball_dets

    if (frame_idx + 1) % CHECKPOINT_EVERY == 0:
        with open(CHECKPOINT_PATH, "wb") as f:
            pickle.dump({"detection_cache": detection_cache, "frame_idx": frame_idx}, f)

    frame_idx += 1
    pbar.update(1)

pbar.close()
cap.release()
print(f"Done. {len(detection_cache)} frames processed.")


Resuming from frame 0 (0 frames already cached)
Video: 70327 frames @ 25.00 fps
Detecting: 100%|██████████| 70327/70327 [6:46:48<00:00,  2.88frame/s]
Done. 70326 frames processed.


In [6]:
import shutil, os

with open(OUTPUT_CACHE_PATH, "wb") as f:
    pickle.dump(detection_cache, f)

size_mb = os.path.getsize(OUTPUT_CACHE_PATH) / 1e6
print(f"Saved {OUTPUT_CACHE_PATH} ({size_mb:.1f} MB)")

zip_path = shutil.make_archive(OUTPUT_CACHE_PATH.replace(".pkl", ""), "zip", ".", OUTPUT_CACHE_PATH)
print(f"Zipped: {zip_path}")


Saved barca_atletico_firsthalf_detection_cache_final.pkl (136.7 MB)
Zipped: /kaggle/working/barca_atletico_firsthalf_detection_cache_final.zip


In [7]:
import numpy as np
from collections import defaultdict

# --- per-class totals ---
conf_by_class = defaultdict(list)
for dets in detection_cache.values():
    for d in dets:
        conf_by_class[d["class"]].append(d["conf"])

print("Per-class totals and confidence distribution:")
for cls, confs in conf_by_class.items():
    arr = np.array(confs)
    print(f"  {cls:12s} n={len(arr):8d}")

# --- ball detection rate ---
frames_with_ball = sum(1 for dets in detection_cache.values() if any(d["class"] == "ball" for d in dets))
print(f"\nFrames with a ball detection: {frames_with_ball}/{len(detection_cache)} ({100*frames_with_ball/len(detection_cache):.1f}%)")

# --- per-frame class counts ---
counts_per_frame = defaultdict(list)
for dets in detection_cache.values():
    per_frame = defaultdict(int)
    for d in dets:
        per_frame[d["class"]] += 1
    for cls in ("player", "goalkeeper", "referee"):
        counts_per_frame[cls].append(per_frame.get(cls, 0))

print("\nPer-frame class count distribution:")
for cls in ("player", "goalkeeper", "referee"):
    arr = np.array(counts_per_frame[cls])
    print(f"  {cls:12s} mean={arr.mean():.2f}  min={arr.min()}  max={arr.max()}")


Per-class totals and confidence distribution:
  ball         n=   43034
  referee      n=  179199
  player       n= 1384089
  goalkeeper   n=   63111

Frames with a ball detection: 43034/70326 (61.2%)

Per-frame class count distribution:
  player       mean=19.68  min=18  max=20
  goalkeeper   mean=0.90  min=0  max=1
  referee      mean=2.55  min=1  max=3
